# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a classification task at its core. For each (client, content) pair, I predict one of four states for a future window: growing, declining, recovering, or stable. It also has a scoring/ranking layer on top: once every page has a predicted class and a confidence score, I sort them into an action queue (highest-confidence declining pages first). So the model is a classifier; the deliverable is a ranking built from its output.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The label is a defined proxy, not something already sitting in the data. I compare a past window (e.g. days 1-3) against a future window (e.g. days 4-5) for a chosen metric — total GSC clicks, falling back to impressions when clicks are near zero, since clicks are sparse in this sample. I turn the percent change into a class with fixed thresholds (example below). This is honest labeling, not measurement: the threshold choice is a modeling decision I have to defend, and it will move to 7/30/90-day windows once the full 18-month dataset is used.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Main metric: "macro-F1" across the four classes, compared against a simple baseline ("last period's direction continues"). Macro-F1 matters more than plain accuracy here because the classes are not balanced — most pages are 'stable' or have too little traffic to move, so a model that just predicts 'stable' every time would score high on accuracy but be useless. Since the real output is a ranked action queue, I also track "precision@20" on the 'declining' class: of the top 20 pages the model flags as declining, how many actually decline in the following window. That is the number that matters to whoever reads the queue.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one (client, content page) pair, aggregated over a time window. This sample file only covers 5 days (2025-01-27 to 2025-01-31) for 2 clients and 476 content pages, so I aggregate over the whole 5-day slice as a stand-in for what a real 7/30/90-day window will look like on the full 18-month dataset.

In [7]:
import pandas as pd
from pathlib import Path

data_path = "../../data/raw/fact_content_daily_performance/data_0.parquet"
assert data_path is not None, "data_0.parquet not found - check the data/ folder path"

df = pd.read_parquet(data_path)
df["report_date"] = pd.to_datetime(df["report_date"])
print("raw rows:", df.shape[0], "| date range:", df.report_date.min().date(), "to", df.report_date.max().date())
print("clients:", df.client_hash_id.nunique(), "| content pages:", df.content_hash_id.nunique())

unit_of_analysis = (
    df.groupby(["client_hash_id", "content_hash_id"])
      .agg(
          days_seen=("report_date", "nunique"),
          total_impressions=("gsc_impressions", "sum"),
          total_clicks=("gsc_clicks", "sum"),
          avg_position=("gsc_avg_position", "mean"),
      )
      .reset_index()
)
print("\none row = one (client, content) pair -> shape:", unit_of_analysis.shape)
unit_of_analysis.head(8)

raw rows: 1297 | date range: 2025-01-27 to 2025-01-31
clients: 2 | content pages: 476

one row = one (client, content) pair -> shape: (476, 6)


,client_hash_id,content_hash_id,days_seen,total_impressions,total_clicks,avg_position
0,client_9958f0a7ae1df715,content_0217be03126aa7a5,5,119,0,6.422527
1,client_9958f0a7ae1df715,content_0642dc7f62d4f780,5,53,4,17.080639
2,client_9958f0a7ae1df715,content_0672d8db776419c0,5,25,1,12.572619
3,client_9958f0a7ae1df715,content_0ca502c18c4fd41e,5,30,0,12.189048
4,client_9958f0a7ae1df715,content_0d308caf94a3ed16,5,87,0,31.006077
5,client_9958f0a7ae1df715,content_0fa49fb87f830edf,5,48,0,31.226719
6,client_9958f0a7ae1df715,content_132cfd61ee6071be,5,78,4,32.885131
7,client_9958f0a7ae1df715,content_1513a75b3809e368,3,9,0,26.805556


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "if impressions go up, the page is growing" breaks down because impressions and position (ranking) do not move together in a clean, predictable way. I checked this directly on the sample below: the day-over-day correlation between impression change and position change is close to zero, and in a meaningful share of day-steps impressions rose while position got *worse* at the same time — a simple rule would call that 'growing' when it's actually mixed or negative signal. Clicks are also sparse (fewer than 11% of content pages had any clicks at all in this 5-day slice), so a rule based on clicks alone would sit silent for most pages. A model can weigh several noisy, sometimes-contradicting signals together and learn client-specific baselines, instead of one threshold applied the same way to every page.

In [8]:
# Evidence: how noisy is the impressions <-> position relationship, and how sparse are clicks?
df_sorted = df.sort_values(["client_hash_id", "content_hash_id", "report_date"])
counts = df_sorted.groupby(["client_hash_id", "content_hash_id"])["report_date"].transform("nunique")
full_history = df_sorted[counts == counts.max()].copy()

full_history["imp_change"] = full_history.groupby(["client_hash_id", "content_hash_id"])["gsc_impressions"].diff()
full_history["pos_change"] = full_history.groupby(["client_hash_id", "content_hash_id"])["gsc_avg_position"].diff()
steps = full_history.dropna(subset=["imp_change", "pos_change"])

corr = steps["imp_change"].corr(steps["pos_change"])
contradiction_rate = ((steps["imp_change"] > 0) & (steps["pos_change"] > 0)).mean()
click_share = (unit_of_analysis["total_clicks"] > 0).mean()

print(f"correlation, impression change vs position change (day-over-day): {corr:.3f}")
print(f"share of day-steps where impressions rose but position got worse at the same time: {contradiction_rate:.1%}")
print(f"share of content pages with any clicks at all in this 5-day slice: {click_share:.1%}")

correlation, impression change vs position change (day-over-day): -0.081
share of day-steps where impressions rose but position got worse at the same time: 20.8%
share of content pages with any clicks at all in this 5-day slice: 10.7%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.